# 08 — Verify Evaluation Pipeline Consistency

This notebook verifies that `src/eval/` produces the same results as the
inline evaluation used in notebooks 03/04/06/07.

**No model inference needed** — we re-evaluate the saved prediction strings
from existing result files through the `src/eval/` pipeline.

Expected results to match:

| Source | Model | Notebook Acc |
|---|---|---|
| `controlled_comparison_results.json` | Base (raw prompt) | 37.2% |
| `controlled_comparison_results.json` | V1-fixed (chat prefix) | 51.8% |
| `base_model_results.json` | Base (original eval) | 37.2% |
| `v1_fixed_eval_results.json` | V1-fixed (notebook 04) | 51.4% |
| `base_model_chat_prefix.json` | Base + chat prefix | 1.8% |

In [2]:
!pip install -q "datasets<3.0"

In [3]:
import json
import sqlite3
from datasets import load_dataset

# Load the same 500 test examples used in all experiments
ds = load_dataset("Salesforce/wikisql", trust_remote_code=True)
examples = list(ds['test'].select(range(500)))
print(f"Loaded {len(examples)} test examples")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating test split:   0%|          | 0/15878 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8421 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/56355 [00:00<?, ? examples/s]

Loaded 500 test examples


## Step 1: Define `src/eval/` pipeline

Copy the exact code from `src/data/sql_executor.py`, `src/data/prepare_dataset.py`,
and `src/eval/execution_accuracy.py` here so this notebook is self-contained
in Colab. **Do not modify anything** — the point is to test the `src/` code as-is.

In [4]:
# ── src/data/prepare_dataset.py (gold SQL builder) ─────────────────────

AGG_OPS = ["", "MAX", "MIN", "COUNT", "SUM", "AVG"]
COND_OPS = ["=", ">", "<"]

def build_sql_string(sql: dict, columns: list, types: list = None) -> str:
    """Convert WikiSQL structured SQL dict to a human-readable SQL string."""
    agg = AGG_OPS[sql["agg"]]
    sel_col = columns[sql["sel"]]
    if agg:
        select_clause = f"SELECT {agg}(`{sel_col}`)"
    else:
        select_clause = f"SELECT `{sel_col}`"
    where_clauses = []
    for col_idx, op_idx, cond in zip(
        sql["conds"]["column_index"],
        sql["conds"]["operator_index"],
        sql["conds"]["condition"],
    ):
        col = columns[col_idx]
        op = COND_OPS[op_idx]
        col_type = types[col_idx] if types else "text"
        if col_type in ("real", "number"):
            cleaned_cond = str(cond).replace(",", "")
            try:
                float(cleaned_cond)
                where_clauses.append(f"`{col}` {op} {cleaned_cond}")
            except ValueError:
                escaped_cond = str(cond).replace("'", "''")
                where_clauses.append(f"`{col}` {op} '{escaped_cond}'")
        else:
            escaped_cond = str(cond).replace("'", "''")
            where_clauses.append(f"`{col}` {op} '{escaped_cond}'")
    if where_clauses:
        where_clause = " WHERE " + " AND ".join(where_clauses)
    else:
        where_clause = ""
    return select_clause + " FROM table" + where_clause

print("build_sql_string defined (from src/data/prepare_dataset.py)")

build_sql_string defined (from src/data/prepare_dataset.py)


In [5]:
# ── src/data/sql_executor.py ──────────────────────────────────────────

def build_db(table: dict):
    """Create an in-memory SQLite database from a WikiSQL table."""
    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()
    headers = table["header"]
    types = table["types"]
    type_map = {"text": "TEXT", "number": "REAL", "real": "REAL"}
    col_defs = ", ".join(
        f'`{col}` {type_map.get(t, "TEXT")}'
        for col, t in zip(headers, types)
    )
    cursor.execute(f"CREATE TABLE data ({col_defs})")
    for row in table["rows"]:
        converted = []
        for val, col_type in zip(row, types):
            if col_type in ("real", "number"):
                try:
                    converted.append(float(val))
                except (ValueError, TypeError):
                    converted.append(val)
            else:
                converted.append(val)
        placeholders = ", ".join(["?"] * len(converted))
        cursor.execute(f"INSERT INTO data VALUES ({placeholders})", converted)
    conn.commit()
    return conn


def execute_sql(table: dict, sql: str):
    """Execute a SQL query against an in-memory SQLite DB."""
    sql = sql.replace("FROM table", "FROM data")
    try:
        conn = build_db(table)
        cursor = conn.cursor()
        cursor.execute(sql)
        results = cursor.fetchall()
        conn.close()
        return results, None
    except Exception as e:
        return [], str(e)

print("execute_sql defined (from src/data/sql_executor.py)")
print("NOTE: This is the CURRENT src/ version. Check if float(val) handles commas.")

execute_sql defined (from src/data/sql_executor.py)
NOTE: This is the CURRENT src/ version. Check if float(val) handles commas.


In [6]:
# ── src/eval/execution_accuracy.py ────────────────────────────────────

def normalize_result(result: list) -> list:
    """Sort and stringify results for comparison."""
    return sorted([str(row) for row in result])


def evaluate_single(predicted_sql: str, gold_sql: dict, table: dict) -> dict:
    gold_sql_str = build_sql_string(gold_sql, table["header"], table["types"])
    pred_results, pred_error = execute_sql(table, predicted_sql)
    gold_results, gold_error = execute_sql(table, gold_sql_str)
    if pred_error:
        return {"correct": False, "syntax_error": True, "error_message": pred_error}
    correct = normalize_result(pred_results) == normalize_result(gold_results)
    return {"correct": correct, "syntax_error": False, "error_message": None}


def evaluate_dataset(predictions: list, dataset: list) -> dict:
    total = len(predictions)
    correct = 0
    syntax_errors = 0
    for pred, example in zip(predictions, dataset):
        result = evaluate_single(pred, example["sql"], example["table"])
        if result["correct"]:
            correct += 1
        if result["syntax_error"]:
            syntax_errors += 1
    return {
        "execution_accuracy": correct / total,
        "syntax_error_rate": syntax_errors / total,
        "total": total,
        "correct": correct,
        "syntax_errors": syntax_errors,
    }

print("evaluate_dataset defined (from src/eval/execution_accuracy.py)")

evaluate_dataset defined (from src/eval/execution_accuracy.py)


## Step 2: Sanity check — gold SQL should get 100%

In [7]:
# Generate gold SQL for all 500 examples and evaluate
gold_predictions = [
    build_sql_string(ex["sql"], ex["table"]["header"], ex["table"]["types"])
    for ex in examples
]

sanity = evaluate_dataset(gold_predictions, examples)
print(f"Sanity check (gold SQL as predictions):")
print(f"  Execution accuracy: {sanity['execution_accuracy']:.1%}")
print(f"  Syntax error rate:  {sanity['syntax_error_rate']:.1%}")
print()

if sanity['execution_accuracy'] < 1.0:
    print(f"WARNING: {sanity['total'] - sanity['correct']} gold SQL queries failed!")
    print("This means src/eval has a bug. Check comma handling in sql_executor.py.")
    # Show first few failures
    for pred, ex in zip(gold_predictions, examples):
        r = evaluate_single(pred, ex['sql'], ex['table'])
        if not r['correct']:
            print(f"  Failed: {pred[:80]}")
            if r['error_message']:
                print(f"  Error:  {r['error_message']}")
            break
else:
    print("PASS — src/eval pipeline achieves 100% on gold SQL.")

Sanity check (gold SQL as predictions):
  Execution accuracy: 100.0%
  Syntax error rate:  0.0%

PASS — src/eval pipeline achieves 100% on gold SQL.


## Step 3: Upload result files and re-evaluate

Upload these files from your `results/` folder:
- `controlled_comparison_results.json`
- `base_model_results.json`
- `v1_fixed_eval_results.json`
- `base_model_chat_prefix.json`

In [8]:
from google.colab import files
uploaded = files.upload()

Saving controlled_comparison_results.json to controlled_comparison_results.json
Saving base_model_results.json to base_model_results.json
Saving v1_fixed_eval_results.json to v1_fixed_eval_results.json
Saving base_model_chat_prefix.json to base_model_chat_prefix.json


In [9]:
def load_predictions(filename):
    """Load prediction SQL strings from a result file."""
    with open(filename) as f:
        data = json.load(f)
    preds = data.get('predictions', [])
    if not preds:
        return []
    # Handle both formats: list of strings, or list of dicts with 'predicted_sql'
    if isinstance(preds[0], str):
        return preds
    elif isinstance(preds[0], dict):
        return [p['predicted_sql'] for p in preds]
    return []


def load_comparison_predictions(filename, model_key):
    """Load predictions from a controlled_comparison_results.json file."""
    with open(filename) as f:
        data = json.load(f)
    return data[model_key]['predictions']


print("Loader functions defined")

Loader functions defined


In [10]:
# ── Re-evaluate all saved predictions through src/eval ──

results_to_check = [
    {
        "name": "Controlled: Base (raw prompt)",
        "expected_acc": 37.2,
        "loader": lambda: load_comparison_predictions('controlled_comparison_results.json', 'base_model'),
    },
    {
        "name": "Controlled: V1-fixed (chat prefix)",
        "expected_acc": 51.8,
        "loader": lambda: load_comparison_predictions('controlled_comparison_results.json', 'v1_fixed'),
    },
    {
        "name": "Original baseline (notebook 03)",
        "expected_acc": 37.2,
        "loader": lambda: load_predictions('base_model_results.json'),
    },
    {
        "name": "V1-fixed (notebook 04)",
        "expected_acc": 51.4,
        "loader": lambda: load_predictions('v1_fixed_eval_results.json'),
    },
    {
        "name": "Base + chat prefix (notebook 07)",
        "expected_acc": 1.8,
        "loader": lambda: load_predictions('base_model_chat_prefix.json'),
    },
]

print("=" * 75)
print("EVALUATION PIPELINE CONSISTENCY CHECK")
print("=" * 75)
print(f"{'Source':<42} {'Expected':>9} {'src/eval':>9} {'Match':>7}")
print("-" * 75)

all_pass = True
for item in results_to_check:
    preds = item['loader']()
    if len(preds) != 500:
        print(f"{item['name']:<42} {'SKIP — ' + str(len(preds)) + ' predictions':>25}")
        continue
    metrics = evaluate_dataset(preds, examples)
    actual = metrics['execution_accuracy'] * 100
    expected = item['expected_acc']
    match = abs(actual - expected) < 0.1
    status = 'PASS' if match else 'DIFF'
    if not match:
        all_pass = False
    print(f"{item['name']:<42} {expected:>8.1f}% {actual:>8.1f}% {status:>7}")

print("-" * 75)
if all_pass:
    print("\nALL PASS — src/eval is consistent with inline evaluation.")
    print("You can safely use src/eval as the canonical pipeline.")
else:
    print("\nSOME DIFFERENCES FOUND — investigate before standardizing.")
    print("Likely cause: comma handling in sql_executor.py build_db().")

EVALUATION PIPELINE CONSISTENCY CHECK
Source                                      Expected  src/eval   Match
---------------------------------------------------------------------------
Controlled: Base (raw prompt)                  37.2%     37.2%    PASS
Controlled: V1-fixed (chat prefix)             51.8%     51.8%    PASS
Original baseline (notebook 03)                37.2%     37.2%    PASS
V1-fixed (notebook 04)                         51.4%     51.4%    PASS
Base + chat prefix (notebook 07)                1.8%      0.0%    DIFF
---------------------------------------------------------------------------

SOME DIFFERENCES FOUND — investigate before standardizing.
Likely cause: comma handling in sql_executor.py build_db().


## Step 4: Detailed diff (only if there are mismatches)

Run this cell only if any result showed DIFF above.
It identifies which specific predictions changed outcome.

In [11]:
# Pick whichever result had a DIFF (change the loader as needed)
# Default: controlled comparison base model
preds = load_comparison_predictions('controlled_comparison_results.json', 'base_model')

diffs = []
for i, (pred, ex) in enumerate(zip(preds, examples)):
    r = evaluate_single(pred, ex['sql'], ex['table'])
    # We don't know the inline result per-example, but we can flag errors
    if r['syntax_error']:
        # Check if the error is comma-related
        if r['error_message'] and (',' in str(ex['table']['rows'][:1])):
            diffs.append({
                'index': i,
                'pred': pred[:80],
                'error': r['error_message'][:80],
                'has_comma_values': True
            })

if diffs:
    print(f"Found {len(diffs)} potential comma-related errors:")
    for d in diffs[:5]:
        print(f"  [{d['index']}] {d['pred']}")
        print(f"         Error: {d['error']}")
else:
    print("No comma-related errors found.")

Found 109 potential comma-related errors:
  [14] SELECT Ship FROM (   SELECT Ship, `Launched`, ROW_NUMBER() OVER (PARTITION BY Sh
         Error: no such column: Ship
  [25] SELECT Rank FROM table WHERE `Wrestler` = 'Bryan Danielson'
         Error: no such column: Rank
  [27] SELECT Rank FROM table WHERE `Wrestler` = 'Go Shiozaki'
         Error: no such column: Rank
  [30] SELECT COUNT(*) FROM table WHERE CAST(SUBSTRING_INDEX(`Singles W–L`, '-', 1) AS 
         Error: no such function: SUBSTRING_INDEX
  [34] SELECT COUNT(*) FROM table WHERE CAST(`Total W–L` AS INT) = 38 AND CAST(SUBSTRIN
         Error: near "FROM": syntax error
